# Dự đoán thời gian chuyến taxi New York – Đội UNETI18
Olympic AI UNETI 2026 – Vòng 2

## 1. Chuẩn bị

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/uneti18_taxi'
TRAIN_FILE  = 'data/train.csv'
TEAM_TEST   = 'data/UNETI18_TEST_3000.csv'
TEST_DIR    = 'data/test'          # thư mục chứa các bộ test khác
PRED_DIR    = 'outputs/predictions'

from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir(PROJECT_DIR)

In [ ]:
!pip install -q -r requirements.txt

## 2. Huấn luyện
EDA, 4 mô hình (baseline, Ridge, LightGBM, LightGBM + target encoding), chọn mô hình theo RMSLE kiểm định.

In [ ]:
!python train.py --train {TRAIN_FILE} --test {TEAM_TEST} --seeds 3

In [ ]:
import json, pandas as pd
m = json.load(open('outputs/metrics.json', encoding='utf-8'))
print('Mô hình chọn:', m['final_model'])
print('Giới hạn nhiễu ước tính:', round(m['error_analysis']['noise_floor'], 4))
pd.DataFrame(m['results'])[['model', 'rmsle_val', 'diem_AI_uoc_tinh']]

## 3. Đánh giá trên các bộ test giả
Các file 3.000 dòng cắt từ tập kiểm định (có nhãn), chấm bằng mô hình chỉ học 80% dữ liệu.

In [ ]:
!python predict.py --test data/pseudo_tests --artifacts artifacts_val --out-dir outputs/pseudo_eval

## 4. Dự đoán bộ test

In [ ]:
!python predict.py --test {TEAM_TEST} --out outputs/submission_UNETI18.csv --out-dir outputs
!python predict.py --test {TEST_DIR} --out-dir {PRED_DIR}

In [ ]:
pd.read_csv(f'{PRED_DIR}/tong_hop.csv')

## 5. Báo cáo và xuất kết quả

In [ ]:
!python make_report.py
from IPython.display import Image, display
for f in ['fig1_duration_dist', 'fig2_heatmap_dow_hour', 'fig3_distance_vs_duration', 'fig8_model_compare']:
    display(Image(f'outputs/eda/{f}.png'))

In [ ]:
# Đổi encoder sang .npz (nếu còn bản .joblib cũ) rồi nén để chạy ứng dụng trên máy
import sys, joblib
sys.path.insert(0, 'src')
for d in ['artifacts', 'artifacts_val']:
    if os.path.exists(f'{d}/encoder.joblib') and not os.path.exists(f'{d}/encoder.npz'):
        joblib.load(f'{d}/encoder.joblib').save(f'{d}/encoder.npz')

!rm -f ket_qua.zip
!zip -qr ket_qua.zip artifacts artifacts_val outputs data/pseudo_tests
from google.colab import files
files.download('ket_qua.zip')